# Needle 2 robot-DSL fine-tune on Colab

Runtime > Change runtime type > **T4 GPU**. Then:
1. Run cell 1, upload `colab_bundle.tar.gz` when prompted
2. Run cells in order
3. Watch `loss` / `val` each epoch — they must start ~2-4 and DROP. If you see `0.0000`, stop: the config is broken.
4. Cell 5 tests the built model; cell 6 downloads `robot.cact` + adapter

In [ ]:
# 1. upload the bundle
from google.colab import files
up = files.upload()  # choose colab_bundle.tar.gz
!tar xzf colab_bundle.tar.gz
!mkdir -p checkpoints
!wc -l data/finetune/train_v2.jsonl

In [ ]:
# 2. install needle with CUDA jax
!pip install -q "cactus-needle[gpu]"
import needle, jax
print('needle ok, jax devices:', jax.devices())

In [ ]:
# 3. fine-tune (~30-60 min on T4; watch loss drop per epoch)
!needle finetune data/finetune/train_v2.jsonl \
  --epochs 10 --val-split 0.1 \
  --out checkpoints/needle_lora_v2.pkl

In [ ]:
# 4. build the deployable 2-bit model
!needle build checkpoints/needle2.pkl --lora checkpoints/needle_lora_v2.pkl --bits 2 --out robot.cact
!ls -la robot.cact

In [ ]:
# 5. empirical check — outputs must match the labels
import json, needle
schema = json.load(open('schema/tool_schema.json'))
agent = needle.Needle(weights='robot.cact', tools=schema,
                      system='device: domestic robot; locale: en-US')
tests = [
    'go to my room',
    'go to my room and wait there for 5 minutes and then go to oven',
    'wait for five seconds',
    'give John the cup',
    'wake up my daughter',
    'go wash yourself',
    'play alarm.wav then stop',
    'clean the kitchen then show dinner is ready',
]
for q in tests:
    r = agent.complete(q)
    print(repr(q))
    print('  ->', json.dumps(r['function_calls']))

# scored spot-check against generated data (first 100 examples)
import sys
sys.path.insert(0, '.')
records = [json.loads(l) for l in open('data/finetune/train_v2.jsonl')][:100]
ok = 0
for rec in records:
    pred = agent.complete(rec['query'])['function_calls'] or []
    gold = rec['answers']
    if len(pred) == len(gold) and all(
        p['name'] == g['name'] and p['arguments'] == g['arguments']
        for p, g in zip(pred, gold)):
        ok += 1
print(f'exact match: {ok}/100')
agent.reset()

In [ ]:
# 6. download artifacts back to the Mac
from google.colab import files
files.download('robot.cact')
files.download('checkpoints/needle_lora_v2.pkl')